# Diffusion Models (DDPM) — MNIST Demo (Exercise)



This notebook is a **hands-on demo** of a *Denoising Diffusion Probabilistic Model (DDPM)* trained on **MNIST (28×28 grayscale digits)**.

### What you will learn
- A forward (noising) process: $x_0 \to x_t$
- A neural network $\varepsilon_\theta(x_t, t)$ that predicts the noise added at timestep \(t\)
- A reverse (denoising) sampling loop that generates **new digits from pure noise**


We use MNIST dataset. MNIST trains quickly in Colab and still lets you see the diffusion pipeline end-to-end.

You will implement the missing parts marked **TODO**.
---


In [ ]:

# --- Colab-friendly setup (PyTorch + torchvision are typically preinstalled) ---
import math
import os
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

def set_seed(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:

# --- MNIST data ---
# We normalize to [-1, 1] which is common for diffusion models.
transform = transforms.Compose([
    transforms.ToTensor(),                 # [0,1]
    transforms.Lambda(lambda x: x * 2 - 1)  # [-1,1]
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
)

# Quick peek
x0, y0 = next(iter(train_loader))
print("Batch:", x0.shape, y0.shape, "min/max:", float(x0.min()), float(x0.max()))


In [ ]:

def show_images(batch, title=None, n=16):
    # batch: Tensor [B,1,28,28] in [-1,1] or [0,1].
    batch = batch.detach().cpu()
    if batch.min() < -0.5:  # assume [-1,1] -> map to [0,1]
        batch = (batch + 1) / 2
    batch = torch.clamp(batch, 0, 1)

    n = min(n, batch.size(0))
    cols = int(math.sqrt(n))
    rows = int(math.ceil(n / cols))

    fig = plt.figure(figsize=(cols * 2, rows * 2))
    if title:
        fig.suptitle(title)
    for i in range(n):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(batch[i, 0], cmap="gray")
        ax.axis("off")
    plt.show()



## 1) Diffusion math

We define a noise schedule \\(\{\beta_t\}_{t=0}^{T-1}\\) with small positive values.

- $\alpha_t = 1 - \beta_t$
- $\bar\alpha_t = \prod_{s=0}^t \alpha_s$

**Forward process** (closed form):
$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon,\quad \varepsilon\sim\mathcal N(0,I)$

**Training objective**:
Sample a random timestep \\(t\\), form \\(x_t\\), and train a network to predict \\(\varepsilon\\):
$
\min_\theta\ \mathbb E_{x_0,t,\varepsilon}\left[\|\varepsilon - \varepsilon_\theta(x_t,t)\|_2^2\right]
$

**Reverse sampling step**:
$
x_{t-1} = \mu_\theta(x_t,t) + \sigma_t z,\quad z\sim\mathcal N(0,I)
$
where
$
\mu_\theta(x_t,t)=\frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\varepsilon_\theta(x_t,t)\right),\quad
\sigma_t=\sqrt{\beta_t}$ and for \\(t=0\\) we set \\(z=0\\).

---



## 2) A tiny U-Net

To keep the exercise focused, the U-Net is given.
It takes `(x_t, t)` and predicts noise `eps_pred` with the same shape as `x_t`.

You can try to build your own one.


In [ ]:

# --- Time embedding utilities ---
def sinusoidal_embedding(timesteps: torch.Tensor, dim: int):
    # TODO (EXERCISE):
    # Implement standard sinusoidal timestep embeddings.
    # - timesteps: int64 tensor [B] (values in [0, T-1])
    # - returns: float32 tensor [B, dim]
    # Hints:
    #   * Use half dim for sin, half for cos.
    #   * Use log-spaced frequencies.
    raise NotImplementedError("TODO: implement sinusoidal_embedding")


class TimeMLP(nn.Module):
    def __init__(self, emb_dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t_emb):
        return self.net(t_emb)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_ch, out_ch)
        self.act = nn.SiLU()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)

    def forward(self, x, t_feat):
        h = self.conv1(x)
        h = self.norm1(h)
        h = self.act(h)

        # Add time feature (FiLM-lite)
        t_add = self.time_proj(t_feat)[:, :, None, None]
        h = h + t_add

        h = self.conv2(h)
        h = self.norm2(h)
        h = self.act(h)

        return h + self.skip(x)


class Down(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.pool = nn.Conv2d(ch, ch, 4, stride=2, padding=1)  # 28->14->7

    def forward(self, x):
        return self.pool(x)


class Up(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)

    def forward(self, x):
        return self.up(x)


class TinyUNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=64, time_emb_dim=64):
        super().__init__()
        self.time_emb_dim = time_emb_dim
        self.time_mlp = TimeMLP(time_emb_dim, hidden_dim=base_ch)

        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        self.rb1 = ResBlock(base_ch, base_ch, time_ch=base_ch)
        self.down1 = Down(base_ch)
        self.rb2 = ResBlock(base_ch, base_ch * 2, time_ch=base_ch)

        self.down2 = Down(base_ch * 2)
        self.mid = ResBlock(base_ch * 2, base_ch * 2, time_ch=base_ch)

        self.up1 = Up(base_ch * 2)
        self.rb3 = ResBlock(base_ch * 2 + base_ch * 2, base_ch, time_ch=base_ch)

        self.up2 = Up(base_ch)
        self.rb4 = ResBlock(base_ch + base_ch, base_ch, time_ch=base_ch)

        self.out = nn.Conv2d(base_ch, 1, 3, padding=1)

    def forward(self, x, t):
        # t: [B] int64
        t_emb = sinusoidal_embedding(t, self.time_emb_dim).to(x.dtype).to(x.device)
        t_feat = self.time_mlp(t_emb)  # [B, base_ch]

        x1 = self.in_conv(x)
        x1 = self.rb1(x1, t_feat)      # [B, base_ch, 28, 28]

        x2 = self.down1(x1)            # [B, base_ch, 14, 14]
        x2 = self.rb2(x2, t_feat)      # [B, 2*base_ch, 14, 14]

        x3 = self.down2(x2)            # [B, 2*base_ch, 7, 7]
        x3 = self.mid(x3, t_feat)      # [B, 2*base_ch, 7, 7]

        u1 = self.up1(x3)              # [B, 2*base_ch, 14, 14]
        u1 = torch.cat([u1, x2], dim=1)
        u1 = self.rb3(u1, t_feat)      # [B, base_ch, 14, 14]

        u2 = self.up2(u1)              # [B, base_ch, 28, 28]
        u2 = torch.cat([u2, x1], dim=1)
        u2 = self.rb4(u2, t_feat)      # [B, base_ch, 28, 28]

        return self.out(u2)            # predicted noise, same shape as x


In [ ]:

## 3) Diffusion schedule + forward process (EXERCISE)

def get_linear_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02):
    # TODO (EXERCISE): Implement a simple linear beta schedule.
    # Returns: betas tensor of shape [T] (float32).
    raise NotImplementedError("TODO: implement get_linear_beta_schedule")


def extract(a: torch.Tensor, t: torch.Tensor, x_shape):
    # TODO (EXERCISE): Extract coefficients for a batch of timesteps.
    # - a: [T]
    # - t: [B] int64
    # - returns: [B,1,1,1] broadcastable to x_shape
    # Hint: use torch.gather and reshape.
    raise NotImplementedError("TODO: implement extract")


def q_sample(x0: torch.Tensor, t: torch.Tensor, sqrt_alphas_cumprod: torch.Tensor,
             sqrt_one_minus_alphas_cumprod: torch.Tensor, noise: torch.Tensor = None):
    # TODO (EXERCISE): x_t = sqrt(ab_t)*x0 + sqrt(1-ab_t)*noise
    raise NotImplementedError("TODO: implement q_sample")


In [ ]:

## 4) Training (EXERCISE)

@dataclass
class TrainConfig:
    T: int = 100
    lr: float = 1e-3
    base_channels: int = 64
    num_train_steps: int = 100   # optimizer steps; adjust for speed/quality
    grad_clip: float = 1.0

cfg = TrainConfig()

# Precompute schedule terms
betas = get_linear_beta_schedule(cfg.T).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

model = TinyUNet(in_ch=1, base_ch=cfg.base_channels, time_emb_dim=64).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)

# optional: mixed precision
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

def train_step(x0):
    # TODO (EXERCISE): Implement one DDPM training step.
    # Steps:
    #  1) Sample random t for each item
    #  2) Sample noise eps ~ N(0, I)
    #  3) Form x_t via q_sample
    #  4) eps_pred = model(x_t, t)
    #  5) loss = MSE(eps_pred, eps)
    raise NotImplementedError("TODO: implement train_step")


# Training loop
model.train()
step = 0
loss_hist = []

while step < cfg.num_train_steps:
    for x0, _ in train_loader:
        x0 = x0.to(device)

        loss = train_step(x0)
        loss_hist.append(loss)

        if step % 10 == 0:
            print(f"step {step:4d} | loss {loss:.4f}")
        step += 1
        if step >= cfg.num_train_steps:
            break

print("Done. Last loss:", loss_hist[-1])


In [ ]:

## 5) Sampling (EXERCISE)

@torch.no_grad()
def p_sample_step(model, x, t, betas, alphas, alphas_cumprod):
    # TODO (EXERCISE): Implement one reverse DDPM step x_t -> x_{t-1}.
    # Use:
    #   mu = 1/sqrt(alpha_t) * ( x - beta_t/sqrt(1-ab_t) * eps_theta(x,t) )
    #   sigma = sqrt(beta_t)
    #   x_{t-1} = mu + sigma * z  (z ~ N(0,I), but z=0 if t==0)
    raise NotImplementedError("TODO: implement p_sample_step")


@torch.no_grad()
def sample_loop(model, n=16, T=100):
    # TODO (EXERCISE): Full sampling loop starting from pure noise.
    # Returns images in [-1,1].
    raise NotImplementedError("TODO: implement sample_loop")


# Generate a batch of samples
model.eval()
samples = sample_loop(model, n=25, T=cfg.T)
show_images(samples, title="Generated samples (after training)", n=25)



## 6) Bonus: Denoise a *given* image

A diffusion model can also **denoise** a noisy input.
We take a real MNIST digit `x0`, add noise to get `x_t`, then run the reverse process to (approximately) recover a digit.


In [ ]:

@torch.no_grad()
def denoise_from_xt(model, x_t, t_start, betas, alphas, alphas_cumprod):
    # Denoise starting from a given x_t at timestep t_start down to 0.
    # You can re-use p_sample_step.
    x = x_t
    for t in reversed(range(t_start + 1)):
        t_batch = torch.full((x.size(0),), t, device=x.device, dtype=torch.long)
        x = p_sample_step(model, x, t_batch, betas, alphas, alphas_cumprod)
    return x

# Pick a random real image, noise it, then denoise it
x0, _ = next(iter(train_loader))
x0 = x0[:16].to(device)

t_start = int(cfg.T * 0.7)
t = torch.full((x0.size(0),), t_start, device=device, dtype=torch.long)
noise = torch.randn_like(x0)
x_t = q_sample(x0, t, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod, noise=noise)

show_images(x0, title="Original x0", n=16)
show_images(x_t, title=f"Noised x_t (t={t_start})", n=16)

x_rec = denoise_from_xt(model, x_t, t_start, betas, alphas, alphas_cumprod)
show_images(x_rec, title="Denoised result (reverse process)", n=16)
